# Deterministic NG-SMR + HB + CCS ammonia NPV

Calculate the expected-input NPV, levelized net margin, and LCOA for NG-SMR + HB + CCS at 1,000,000 tNH3/year using the ammonia source model. The tables show resolved inputs and financial outputs without repeating the calculation formulas. The CCS increment is resolved against NG-SMR + HB. The notebook shows both parent inputs and incremental changes; transport and storage follow the shared CCS cost rule.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ammonia.ammonia_npv_deterministic import calculate_deterministic_ammonia_result
from ammonia.ammonia_npv_summary_figures import (
    AMMONIA_PROCESSED_OUTPUT_COLUMNS,
    AMMONIA_RAW_INPUT_COLUMNS,
    AMMONIA_TECHNOLOGY_LABELS,
)

pd.options.display.float_format = "{:,.3f}".format

In [2]:
TECHNOLOGY = "ng_smr_hb_ccs"
result = calculate_deterministic_ammonia_result(TECHNOLOGY)
values = {key: item[0] for key, item in result.items()}

UNITS = {
    "annual_output_tnh3": "tNH3/year",
    "lifetime_years": "years",
    "capex_eur_per_tnh3": "EUR/(tNH3/year)",
    "fixed_opex_eur_per_tnh3": "EUR/tNH3",
    "variable_opex_eur_per_tnh3": "EUR/tNH3",
    "natural_gas_consumption_mwh_per_tnh3": "MWh/tNH3",
    "coal_consumption_mwh_per_tnh3": "MWh/tNH3",
    "biomass_consumption_mwh_per_tnh3": "MWh/tNH3",
    "electricity_consumption_mwh_per_tnh3": "MWh/tNH3",
    "emissions_tco2_per_tnh3": "tCO2/tNH3",
    "gas_price_eur_per_mwh_th": "EUR/MWh_th",
    "coal_price_eur_per_mwh_th": "EUR/MWh_th",
    "biomass_price_eur_per_mwh_th": "EUR/MWh_th",
    "electricity_price_eur_per_mwh": "EUR/MWh",
    "ammonia_price_eur_per_tnh3": "EUR/tNH3",
    "carbon_price_eur_per_t": "EUR/tCO2",
    "transport_and_storage_share_of_capture_cost": "fraction",
    "transport_and_storage_cost_eur_per_tnh3": "EUR/tNH3",
    "capture_cost_excluding_transport_and_storage_eur_per_tnh3": "EUR/tNH3",
    "discounted_lifetime_output_tnh3": "discounted tNH3",
    "lcoa_eur_per_tnh3": "EUR/tNH3",
    "levelized_net_margin_eur_per_tnh3": "EUR/tNH3",
}

def unit_for(key):
    key = key.removeprefix("bau_").replace("_change_", "_")
    if key.endswith("_reduction_fraction"):
        return "fraction"
    if key in UNITS:
        return UNITS[key]
    if key.endswith("_eur"):
        return "EUR"
    if key in {"technology", "technology_type", "retrofit_bau_mode", "fuel_type"}:
        return "category"
    return ""

def as_table(keys):
    return pd.DataFrame(
        {"Input / output": key, "Value": values[key], "Unit": unit_for(key)}
        for key in keys if key in values and key != "run_id"
    )

summary = pd.DataFrame([
    {"Metric": "Technology", "Value": AMMONIA_TECHNOLOGY_LABELS[TECHNOLOGY], "Unit": ""},
    {"Metric": "Annual ammonia output", "Value": values["annual_output_tnh3"], "Unit": "tNH3/year"},
    {"Metric": "Direct emissions", "Value": values["emissions_tco2_per_tnh3"], "Unit": "tCO2/tNH3"},
    {"Metric": "NPV", "Value": values["npv_eur"] / 1_000_000, "Unit": "million EUR"},
    {"Metric": "Levelized net margin", "Value": values["levelized_net_margin_eur_per_tnh3"], "Unit": "EUR/tNH3"},
    {"Metric": "LCOA", "Value": values["lcoa_eur_per_tnh3"], "Unit": "EUR/tNH3"},
])
raw_inputs = as_table(AMMONIA_RAW_INPUT_COLUMNS)
processed_outputs = as_table(AMMONIA_PROCESSED_OUTPUT_COLUMNS)
retrofit_keys = [
    key for key in values
    if key.startswith("bau_") or "_change_" in key or key == "emissions_reduction_fraction"
]
retrofit_inputs = as_table(retrofit_keys)

## Summary

In [3]:
summary

,Metric,Value,Unit
0,Technology,NG-SMR + HB + CCS,
1,Annual ammonia output,"1,000,000.000",tNH3/year
2,Direct emissions,0.202,tCO2/tNH3
3,NPV,"3,528.157",million EUR
4,Levelized net margin,330.513,EUR/tNH3
5,LCOA,559.487,EUR/tNH3


## Expected inputs

In [4]:
raw_inputs

,Input / output,Value,Unit
0,technology,ng_smr_hb_ccs,category
1,technology_type,retrofit,category
2,retrofit_bau_mode,deterministic,category
3,annual_output_tnh3,"1,000,000.000",tNH3/year
4,lifetime_years,25,years
5,capex_eur_per_tnh3,"1,265.000",EUR/(tNH3/year)
6,fixed_opex_eur_per_tnh3,50.400,EUR/tNH3
7,variable_opex_eur_per_tnh3,8.950,EUR/tNH3
8,fuel_type,natural_gas,category
9,natural_gas_consumption_mwh_per_tnh3,8.483,MWh/tNH3


## Parent and incremental CCS inputs

In [5]:
retrofit_inputs

,Input / output,Value,Unit
0,bau_capex_eur_per_tnh3,"1,190.000",EUR/(tNH3/year)
1,bau_fixed_opex_eur_per_tnh3,29.400,EUR/tNH3
2,bau_variable_opex_eur_per_tnh3,8.950,EUR/tNH3
3,bau_natural_gas_consumption_mwh_per_tnh3,8.483,MWh/tNH3
4,bau_electricity_consumption_mwh_per_tnh3,0.000,MWh/tNH3
5,bau_emissions_tco2_per_tnh3,1.730,tCO2/tNH3
6,capex_change_eur_per_tnh3,75.000,EUR/(tNH3/year)
7,fixed_opex_change_eur_per_tnh3,21.000,EUR/tNH3
8,variable_opex_change_eur_per_tnh3,0.000,EUR/tNH3
9,natural_gas_consumption_change_mwh_per_tnh3,0.000,MWh/tNH3


## Processed outputs

In [6]:
processed_outputs

,Input / output,Value,Unit
0,technology,ng_smr_hb_ccs,category
1,technology_type,retrofit,category
2,retrofit_bau_mode,deterministic,category
3,initial_capex_eur,"1,265,000,000.000",EUR
4,annual_revenue_eur,"890,000,000.000",EUR
5,annual_fixed_opex_eur,"50,400,000.000",EUR
6,annual_variable_opex_eur,"8,950,000.000",EUR
7,annual_natural_gas_cost_eur,"333,395,000.000",EUR
8,annual_coal_cost_eur,0.000,EUR
9,annual_biomass_cost_eur,0.000,EUR
